# Лабораторна 8. Завдання 5 — порівняльний аналіз

Цей ноутбук **нічого не навчає**. Він читає `results/raw/task1…task4.json`, які
залишили ноутбуки чотирьох мереж, і зводить їх у таблиці та графіки для звіту.

Виконувати **після** всіх чотирьох ноутбуків.

Відповідає на два питання пункту 5:

1. Яка з попередньо навчених мереж краща — і якою ціною?
2. **Чи покращилось розпізнавання після застосування перенесення навчання?**
   Порівняння з мережею, навченою з нуля в лабораторній 7, на тому самому
   розподілі даних і на тих самих власних зображеннях.

Збір JSON-ів чотирьох мереж.

In [ ]:
import json
import zipfile
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

LAB_ROOT = Path("/content") if IN_COLAB else Path.cwd().parent
RESULTS = LAB_ROOT / "results"
RAW = RESULTS / "raw"
RAW.mkdir(parents=True, exist_ok=True)

TASKS = ["task1", "task2", "task3", "task4"]

# У Colab JSON-и беруться з Drive (їх туди складають ноутбуки 1-4);
# локально вони вже лежать у results/raw/ після scripts/unpack_results.py.
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    for name in TASKS:
        src = Path(f"/content/drive/MyDrive/dl_lab8/results/lab8_{name}_results.zip")
        if src.exists():
            with zipfile.ZipFile(src) as z:
                z.extractall(RESULTS)

data = {}
for name in TASKS:
    p = RAW / f"{name}.json"
    if p.exists():
        data[name] = json.loads(p.read_text(encoding="utf-8"))
    else:
        print("НЕМАЄ", p, "— спочатку виконайте відповідний ноутбук")
print("Завантажено:", [d["base"] for d in data.values()])

Базові числа лабораторної 7 — точка відліку.

In [ ]:
# Базові числа лабораторної 7: та сама задача, той самий розподіл 17500/3750/3750,
# але мережа навчена з нуля. Джерело — dl/lab7/results/raw/task4.json.
LAB7 = {
    "base": "Своя CNN з нуля (ЛР7)",
    "params": 1212513,
    "test_accuracy": 0.8424,          # RMSprop(0.001), найкраща з двох у ЛР7
    "test_accuracy_adam": 0.8267,
    "train_time_sec": 1046.4,
    "own": (16, 20),
    "hard": (4, 10),
    "cropped": (2, 6),
}
print(f"{LAB7['base']}: test accuracy {LAB7['test_accuracy']:.4f}, "
      f"{LAB7['params']:,} параметрів")

**Пункт 5.** Зведена таблиця.

In [ ]:
# Пункт 5: зведена таблиця по всіх мережах.
rows = []
for name, d in data.items():
    rows.append({
        "Мережа": d["base"],
        "Параметрів": d["params"]["total"],
        "Навчаних (C)": d["params"]["trainable_stage_c"],
        "Dropout": d["best_dropout"],
        "A: test": d["stage_a"]["test_accuracy"],
        "B: test": d["stage_b"]["test_accuracy"],
        "C: test": d["stage_c"]["test_accuracy"],
        "Час B+C, хв": round((d["stage_b"]["train_time_sec"]
                              + d["stage_c"]["train_time_sec"]) / 60, 1),
        "Модель, МБ": round(d["model"]["size_bytes"] / 1e6, 1),
    })

cols = list(rows[0].keys())
widths = {c: max(len(c), *(len(f"{r[c]:,}" if isinstance(r[c], int) else str(r[c]))
                           for r in rows)) for c in cols}
print(" | ".join(c.ljust(widths[c]) for c in cols))
print("-+-".join("-" * widths[c] for c in cols))
for r in rows:
    print(" | ".join((f"{r[c]:,}" if isinstance(r[c], int) else str(r[c])).ljust(widths[c])
                     for c in cols))

best = max(rows, key=lambda r: r["C: test"])
print(f"\nНайвища точність: {best['Мережа']} — {best['C: test']:.4f}")
print(f"ЛР7 (з нуля):     {LAB7['test_accuracy']:.4f}")
print(f"Приріст:          {best['C: test'] - LAB7['test_accuracy']:+.4f}")

Точність за етапами, з лінією результату мережі «з нуля».

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

names = [d["base"] for d in data.values()]
stage_a = [d["stage_a"]["test_accuracy"] for d in data.values()]
stage_b = [d["stage_b"]["test_accuracy"] for d in data.values()]
stage_c = [d["stage_c"]["test_accuracy"] for d in data.values()]

x = np.arange(len(names))
w = 0.26
fig, ax = plt.subplots(figsize=(11, 5.5))
ax.bar(x - w, stage_a, w, label="Етап A (ознаки)")
ax.bar(x, stage_b, w, label="Етап B (аугментація)")
ax.bar(x + w, stage_c, w, label="Етап C (донавчання)")
ax.axhline(LAB7["test_accuracy"], color="crimson", linestyle="--",
           label=f"ЛР7, мережа з нуля ({LAB7['test_accuracy']:.4f})")

for xi, vals in zip(x, zip(stage_a, stage_b, stage_c)):
    for dx, v in zip((-w, 0, w), vals):
        ax.text(xi + dx, v + 0.004, f"{v:.3f}", ha="center", fontsize=7.5)

ax.set_xticks(x, names)
ax.set_ylabel("Точність на тестовому наборі")
ax.set_ylim(0.8, 1.01)
ax.set_title("Пункт 5: порівняння попередньо навчених мереж за етапами перенесення навчання")
ax.legend(loc="lower right", fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS / "task5_comparison_accuracy.png", dpi=110, bbox_inches="tight")
plt.show()

Скільки коштує точність: параметри проти якості. Тут видно, чи виправдані важкі мережі порівняно з MobileNetV2.

In [ ]:
# Чи варта точність своєї ціни: параметри проти точності.
fig, ax = plt.subplots(figsize=(8.5, 6))
for d in data.values():
    px = d["params"]["total"] / 1e6
    py = d["stage_c"]["test_accuracy"]
    ax.scatter(px, py, s=120)
    ax.annotate(f"{d['base']}\n{px:.1f} млн", (px, py),
                textcoords="offset points", xytext=(8, -4), fontsize=9)
ax.scatter(LAB7["params"] / 1e6, LAB7["test_accuracy"], s=120, color="crimson", marker="X")
ax.annotate(f"ЛР7, з нуля\n{LAB7['params'] / 1e6:.1f} млн",
            (LAB7["params"] / 1e6, LAB7["test_accuracy"]),
            textcoords="offset points", xytext=(8, -4), fontsize=9, color="crimson")
ax.set_xlabel("Параметрів, млн")
ax.set_ylabel("Точність на тестовому наборі (етап C)")
ax.set_title("Ціна точності: розмір мережі проти якості")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS / "task5_comparison_params_vs_acc.png", dpi=110, bbox_inches="tight")
plt.show()

**Пункт 5.** Власні зображення, яких не було в жодному з наборів.

In [ ]:
# Пункт 5: чи покращилось розпізнавання власних зображень?
kinds = [("own_images", "20 звичайних", LAB7["own"]),
         ("hard_images", "10 складних", LAB7["hard"]),
         ("cropped_images", "6 обрізаних", LAB7["cropped"])]

print(f"{'Мережа':<28} | " + " | ".join(f"{label:>14}" for _, label, _ in kinds))
print("-" * 80)
print(f"{LAB7['base']:<28} | " + " | ".join(
    f"{c}/{t:<13}" for _, _, (c, t) in kinds))
own_rows = []
for d in data.values():
    cells = []
    row = {"base": d["base"]}
    for key, _, _ in kinds:
        recs = d[key]
        correct = sum(r["correct"] for r in recs)
        row[key] = [correct, len(recs)]
        cells.append(f"{correct}/{len(recs):<13}")
    own_rows.append(row)
    print(f"{d['base']:<28} | " + " | ".join(cells))

Експорт зведених чисел.

In [ ]:
metrics = {
    "task": "task5_comparison",
    "lab7_baseline": LAB7,
    "models": [
        {
            "base": d["base"],
            "input_size": d["input_size"],
            "params_total": d["params"]["total"],
            "params_trainable_stage_c": d["params"]["trainable_stage_c"],
            "best_dropout": d["best_dropout"],
            "dropout_runs": [{k: r[k] for k in
                              ("p", "best_val_accuracy", "overfit_gap", "final_train_accuracy",
                               "final_val_accuracy")} for r in d["dropout_runs"]],
            "stage_a_test_accuracy": d["stage_a"]["test_accuracy"],
            "stage_a_alt": d["stage_a_alt"],
            "stage_b_test_accuracy": d["stage_b"]["test_accuracy"],
            "stage_c_test_accuracy": d["stage_c"]["test_accuracy"],
            "train_time_sec": d["stage_b"]["train_time_sec"] + d["stage_c"]["train_time_sec"],
            "model_size_bytes": d["model"]["size_bytes"],
            "model_sha256": d["model"]["sha256"],
            "model_file": d["model"]["file"],
        }
        for d in data.values()
    ],
    "own_images_summary": own_rows,
}
out = RAW / "task5.json"
out.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")
print("Збережено", out)

zip_name = "lab8_task5_results.zip"
zip_path = Path("/content") / zip_name if IN_COLAB else RESULTS / zip_name
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(RESULTS.rglob("*")):
        if p.is_file() and "task5" in p.name:
            z.write(p, p.relative_to(RESULTS))
print(zip_path)
if IN_COLAB:
    from google.colab import files

    files.download(str(zip_path))